# Virgin River Forecasting with NARX

In this notebook we forecast river flows on the Virgin River in Utah using the **NARX** (Nonlinear AutoRegressive with eXogenous inputs) model in Flow Forecast. The Virgin River runs through Zion National Park, where flash floods can be dangerous to hikers, so accurate short-term forecasts matter.

NARX predicts the next target value(s) from a window of past target values (the autoregressive lags) plus a window of past exogenous drivers (precipitation, temperature, dew point). It is a lightweight MLP, so it trains quickly and is a strong baseline. At inference time predictions are fed back into the input window (closed-loop) via `simple_decode`.

## 1. Setup: clone Flow Forecast and download data

In [ ]:
from google.colab import auth
import os
auth.authenticate_user()

!git clone https://github.com/AIStream-Peelout/flow-forecast.git
# Single-gage Virgin River file used for this demo
!gsutil cp gs://aistream-datasets/flowdb/09405500AZC_flow.csv .

## 2. Prepare the data

We drop rows missing the timestamp, flow (`cfs`) or precipitation (`p01m`), add a `change_cfs` first-difference column (sometimes easier to learn than the raw level), and write the cleaned frame back to disk for the data loader.

In [ ]:
import pandas as pd

df = pd.read_csv("09405500AZC_flow.csv", low_memory=False)
df = df.dropna(subset=["hour_updated", "cfs", "p01m"])

# The raw timestamps are timezone-aware (e.g. 2014-04-11 16:00:00+00:00). Flow Forecast's
# loaders cast the sort/datetime columns with .astype("datetime64[ns]"), which raises
# "cannot supply both a tz and a timezone-naive dtype" on tz-aware values. We normalize to
# UTC and drop the timezone up front so both the train and test loaders can parse them.
for col in ["hour_updated", "datetime"]:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], utc=True, errors="coerce").dt.tz_localize(None)
df = df.dropna(subset=["hour_updated"])

df = df.sort_values(by=["hour_updated"]).reset_index(drop=True)
df["change_cfs"] = df["cfs"].diff().fillna(0.0)

flow_file_path = "09405500AZC_flow_clean.csv"
df.to_csv(flow_file_path)
print(df.shape)
df[["hour_updated", "cfs", "p01m", "tmpf", "dwpf"]].tail()

## 3. Install Flow Forecast

In [ ]:
os.chdir("flow-forecast")
!pip install -r requirements.txt
!python setup.py develop
os.chdir("..")

## 4. (Optional) Weights & Biases

Flow Forecast can log metrics and plots to W&B. To enable it: uncomment and run the `wandb login` line below, then set `USE_WANDB = True` in the training cell. Logging in only stores your API key (authentication) -- the actual run is created by the `wandb` dict in the config, so the key alone is not enough; you also need `USE_WANDB = True`. Leave it off to run locally with no logging.

In [ ]:
# !wandb login
# os.environ['MODEL_BUCKET'] = "predict_cfs"
# os.environ["ENVIRONMENT_GCP"] = "Colab"
# os.environ["GCP_PROJECT"] = "gmap-997"

## 5. Build the NARX config

Unlike the Informer (which needs the `TemporalLoader` and a decoder), NARX uses the plain `default` `CSVDataLoader`. Key `model_params`:

- **`n_time_series`** — total input columns (targets + exogenous). Here 4: `cfs`, `p01m`, `tmpf`, `dwpf`.
- **`n_targets`** — number of autoregressive targets. Must be the *first* columns of `relevant_cols`. Here 1 (`cfs`).
- **`forecast_history`** — length of the input window.
- **`output_seq_len`** — steps predicted per forward pass; set equal to `forecast_length`.
- **`n_target_lags` / `n_exog_lags`** — how many past steps of the targets / exogenous inputs to feed the MLP (<= `forecast_history`).
- **`hidden_size`, `num_hidden_layers`, `dropout`, `activation`** — MLP capacity.

In [ ]:
FORECAST_HISTORY = 48
FORECAST_LENGTH = 24


def make_narx_config(flow_file_path, weight_path=None, use_wandb=False):
    # Flow Forecast expects params["wandb"] to be a dict (it calls .get() on it) or False.
    # A bare True raises 'bool' object has no attribute 'get'. The dict is what actually
    # creates the run, so logging only happens when use_wandb=True.
    wandb_cfg = False
    if use_wandb:
        wandb_cfg = {
            "name": "narx_virgin",
            "project": "repo-flood_forecast",
            "tags": ["narx", "virgin_river"],
        }
    config = {
        "model_name": "NARX",
        "model_type": "PyTorch",
        "model_params": {
            "n_time_series": 4,
            "forecast_history": FORECAST_HISTORY,
            "output_seq_len": FORECAST_LENGTH,
            "n_targets": 1,
            "n_target_lags": FORECAST_HISTORY,
            "n_exog_lags": FORECAST_HISTORY,
            "hidden_size": 64,
            "num_hidden_layers": 2,
            "dropout": 0.1,
            "activation": "tanh",
        },
        "dataset_params": {
            "class": "default",
            "training_path": flow_file_path,
            "validation_path": flow_file_path,
            "test_path": flow_file_path,
            "batch_size": 64,
            "forecast_history": FORECAST_HISTORY,
            "forecast_length": FORECAST_LENGTH,
            "train_start": 1000,
            "train_end": 50000,
            "valid_start": 50001,
            "valid_end": 57000,
            "test_start": 57000,
            "test_end": 58000,
            "sort_column": "hour_updated",
            "target_col": ["cfs"],
            "relevant_cols": ["cfs", "p01m", "tmpf", "dwpf"],
            "scaler": "StandardScaler",
            "interpolate": {
                "method": "back_forward_generic",
                "params": {"relevant_columns": ["cfs", "p01m", "tmpf", "dwpf"]},
            },
        },
        "training_params": {
            "criterion": "MSE",
            "optimizer": "Adam",
            "optim_params": {},
            "lr": 0.001,
            "epochs": 5,
            "batch_size": 64,
        },
        "early_stopping": {"patience": 2},
        "GCS": False,
        "wandb": wandb_cfg,
        "forward_params": {},
        "metrics": ["MSE"],
        "inference_params": {
            "datetime_start": "2020-05-31",
            "hours_to_forecast": 336,
            "test_csv_path": flow_file_path,
            "decoder_params": {
                "decoder_function": "simple_decode",
                "unsqueeze_dim": 1,
            },
            "dataset_params": {
                "file_path": flow_file_path,
                "sort_column": "hour_updated",
                "forecast_history": FORECAST_HISTORY,
                "forecast_length": FORECAST_LENGTH,
                "relevant_cols": ["cfs", "p01m", "tmpf", "dwpf"],
                "target_col": ["cfs"],
                "scaling": "StandardScaler",
                "interpolate_param": {
                    "method": "back_forward_generic",
                    "params": {"relevant_columns": ["cfs", "p01m", "tmpf", "dwpf"]},
                },
            },
        },
    }
    if weight_path:
        config["weight_path"] = weight_path
    return config

## 6. Train the NARX model

In [ ]:
from flood_forecast.trainer import train_function

# Set to True to log metrics/plots to Weights & Biases (run the `wandb login` cell above
# first). False runs everything locally with no logging. Note: logging in supplies the API
# key but does NOT start a run -- the run is created from the wandb dict in the config, which
# is what use_wandb=True turns on.
USE_WANDB = False

# flow_file_path was set in the data-prep cell; we never left /content so it is still valid.
config = make_narx_config(flow_file_path=flow_file_path, use_wandb=USE_WANDB)
trained_model = train_function("PyTorch", config)

## 7. Inspect a single prediction

Pull one `(x, y)` example from the test loader and run it through the model. NARX returns a tensor of shape `(batch, output_seq_len)` for a single target.

In [ ]:
import torch

x, y = trained_model.test_data[0]
x_in = x.unsqueeze(0).to(trained_model.device).float()
with torch.no_grad():
    pred = trained_model.model(x_in).cpu()
print("input shape :", x_in.shape)
print("output shape:", pred.shape)
print("scaled preds:", pred[0])

## 8. Multi-step closed-loop forecast & plot

`infer_on_torch_model` runs the closed-loop `simple_decode` rollout (feeding predictions back in) and returns predictions alongside the history dataframe, which we plot against the observed flow.

In [ ]:
from flood_forecast.evaluator import infer_on_torch_model
import matplotlib.pyplot as plt

df_pred, end_tensor, hist, idx, test_loader, df_pred_samples = infer_on_torch_model(
    trained_model, **config["inference_params"]
)

plt.figure(figsize=(14, 5))
plt.plot(df_pred["cfs"].values, label="observed cfs")
plt.plot(df_pred["preds"].values, label="NARX forecast")
plt.xlabel("hours")
plt.ylabel("cfs")
plt.title("Virgin River (09405500AZC) — NARX closed-loop forecast")
plt.legend()
plt.show()